In [ ]:
import os
import sys
import ctypes
from pathlib import Path
import numpy as np
import pandas as pd
import napari
import tifffile as tif
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# Make NVRTC/NVIDIA DLLs discoverable before importing CuPy.
if os.name == 'nt':
    torch_lib = Path(sys.prefix) / 'Lib' / 'site-packages' / 'torch' / 'lib'
    dll_dirs = [
        Path(sys.prefix) / 'Library' / 'bin',
        torch_lib,
    ]
    for dll_dir in dll_dirs:
        if dll_dir.exists():
            os.add_dll_directory(str(dll_dir))

    if torch_lib.exists():
        os.environ['PATH'] = f"{torch_lib};" + os.environ.get('PATH', '')
        for name in ('nvrtc64_120_0.dll', 'nvrtc-builtins64_128.dll'):
            dll_path = torch_lib / name
            if dll_path.exists():
                ctypes.WinDLL(str(dll_path))

import cupy as cp
import cupyx.scipy.ndimage as ndi_gpu
import dask.array as da
from skimage.morphology import skeletonize as skeletonize_3d
import sknw
from scipy.spatial.distance import cdist
from scipy.ndimage import distance_transform_edt as edt
from scipy import ndimage as ndi

c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def cupy_chunk_processing(volume, processing_func, chunk_size=(64, 512, 512), overlap=(15, 15, 15), *args, **kwargs):
    result = np.empty_like(volume)
    pool = cp.get_default_memory_pool()
    z_steps = range(0, volume.shape[0], chunk_size[0])

    for z in tqdm(z_steps, desc='Processing chunks'):
        for y in range(0, volume.shape[1], chunk_size[1]):
            for x in range(0, volume.shape[2], chunk_size[2]):
                z_start = max(0, z - overlap[0])
                z_end = min(volume.shape[0], z + chunk_size[0] + overlap[0])
                y_start = max(0, y - overlap[1])
                y_end = min(volume.shape[1], y + chunk_size[1] + overlap[1])
                x_start = max(0, x - overlap[2])
                x_end = min(volume.shape[2], x + chunk_size[2] + overlap[2])

                chunk = volume[z_start:z_end, y_start:y_end, x_start:x_end]
                chunk_gpu = cp.asarray(chunk)

                def func_wrapper(gpu_chunk):
                    gpu_args = []
                    for arg in args:
                        gpu_args.append(cp.asarray(arg) if isinstance(arg, np.ndarray) else arg)

                    gpu_kwargs = {}
                    for k, v in kwargs.items():
                        gpu_kwargs[k] = cp.asarray(v) if isinstance(v, np.ndarray) else v

                    return processing_func(gpu_chunk, *gpu_args, **gpu_kwargs)

                filtered_chunk = func_wrapper(chunk_gpu)

                w_z_start = z - z_start
                w_z_end = w_z_start + chunk_size[0]
                w_y_start = y - y_start
                w_y_end = w_y_start + chunk_size[1]
                w_x_start = x - x_start
                w_x_end = w_x_start + chunk_size[2]

                valid_chunk = filtered_chunk[
                    w_z_start:min(w_z_end, filtered_chunk.shape[0]),
                    w_y_start:min(w_y_end, filtered_chunk.shape[1]),
                    w_x_start:min(w_x_end, filtered_chunk.shape[2]),
                ].get()

                result_z = min(z, result.shape[0])
                result_y = min(y, result.shape[1])
                result_x = min(x, result.shape[2])

                result[
                    result_z:result_z + valid_chunk.shape[0],
                    result_y:result_y + valid_chunk.shape[1],
                    result_x:result_x + valid_chunk.shape[2],
                ] = valid_chunk

                del chunk_gpu, filtered_chunk
                pool.free_all_blocks()

    return result

def measure_edge_length(coordinates):
    # Returns polyline arclength (sum of Euclidean segment lengths).
    differences = np.diff(coordinates, axis=0)
    segment_lengths = np.linalg.norm(differences, axis=1)
    return np.sum(segment_lengths)

def prune_graph(graph, area_3d, edt_cutoff=0.25, length_cutoff=25):
    # Returns a pruned graph after repeatedly removing weak/short endpoint branches.
    # Mathematical: endpoint branch retained if EDT-ratio and length exceed thresholds.
    # Biological: removes likely spurs/noise while keeping stronger vessel termini.
    while True:
        endpoint_nodes = [node for node, degree in graph.degree() if degree == 1]
        values = []
        for node in endpoint_nodes:
            neighbors = list(graph.neighbors(node))

            if len(neighbors) == 1:
                neighbor = neighbors[0]
                edge_data = graph.get_edge_data(neighbor, node)
                edge_length = measure_edge_length(edge_data['pts'])
                branch_edt = area_3d[edge_data['pts'][:, 0], edge_data['pts'][:, 1], edge_data['pts'][:, 2]]
                branch_edt_interp = np.interp(np.linspace(0, 1, 100), np.linspace(0, 1, branch_edt.size), branch_edt)

                if np.mean(branch_edt_interp[:50]) < np.mean(branch_edt_interp[-50:]):
                    neighbor_coords = edge_data['pts'][-1]
                    part_oi = branch_edt_interp[:20]
                else:
                    neighbor_coords = edge_data['pts'][0]
                    part_oi = branch_edt_interp[-20:]

                neighbor_edt = area_3d[neighbor_coords[0], neighbor_coords[1], neighbor_coords[2]]
                value = np.mean(part_oi) / (neighbor_edt + 1e-6)

                if value > edt_cutoff or edge_length <= length_cutoff:
                    graph.remove_node(node)
                    values.append(value)

        if len(values) == 0:
            break
    return graph

def remove_mid_node(graph):
    # Returns graph with degree-2 nodes collapsed into single edges.
    # Mathematical: topological simplification preserving path continuity.
    # Biological: reduces over-segmentation of centerlines into unnecessary intermediate nodes.
    while True:
        nodes_to_process = [n for n, d in graph.degree() if d == 2]
        if not nodes_to_process:
            break

        processed_in_iteration = False
        for i in nodes_to_process:
            if not graph.has_node(i) or graph.degree(i) != 2:
                continue

            neighbors = list(graph.neighbors(i))
            if len(neighbors) != 2:
                continue

            n1, n2 = neighbors[0], neighbors[1]
            if n1 == n2 or graph.has_edge(n1, n2):
                continue

            edge1 = graph.get_edge_data(i, n1)
            edge2 = graph.get_edge_data(i, n2)
            pts1 = np.atleast_2d(edge1['pts'])
            pts2 = np.atleast_2d(edge2['pts'])
            node_coord = graph.nodes[i]['pts'].astype(np.int32)

            s1, e1 = pts1[0], pts1[-1]
            s2, e2 = pts2[0], pts2[-1]

            dists = cdist([s1, e1], [s2, e2])
            min_row, min_col = np.unravel_index(np.argmin(dists), dists.shape)

            if min_row == 0 and min_col == 0:
                combined_line = np.concatenate([pts1[::-1], [node_coord], pts2], axis=0)
            elif min_row == 1 and min_col == 1:
                combined_line = np.concatenate([pts1, [node_coord], pts2[::-1]], axis=0)
            elif min_row == 0 and min_col == 1:
                combined_line = np.concatenate([pts2[::-1], [node_coord], pts1], axis=0)
            else:
                combined_line = np.concatenate([pts1, [node_coord], pts2], axis=0)

            new_weight = edge1.get('weight', 0) + edge2.get('weight', 0)
            graph.add_edge(n1, n2, weight=new_weight, pts=combined_line)
            graph.remove_node(i)
            processed_in_iteration = True

        if not processed_in_iteration:
            break
    return graph

def collect_border_vicinity_edges(graph, image_shape, vicinity_xy=50):
    # Returns subgraph after removing edges near image borders.
    # Biological: downweights partial vessels likely truncated by FOV boundaries.
    border_vicinity_edges = set()
    for u, v in graph.edges():
        try:
            pts = graph[u][v]['pts']
            if any((
                pt[1] < vicinity_xy or pt[1] > image_shape[1] - 1 - vicinity_xy or
                pt[2] < vicinity_xy or pt[2] > image_shape[2] - 1 - vicinity_xy
                ) for pt in pts):
                border_vicinity_edges.add((u, v))
        except KeyError:
            continue

    edges_to_remove = [edge for edge in border_vicinity_edges if graph.has_edge(*edge)]
    graph.remove_edges_from(edges_to_remove)

    isolated_nodes = [node for node in graph.nodes() if graph.degree[node] == 0]
    if isolated_nodes:
        graph.remove_nodes_from(isolated_nodes)

    return graph

def compute_cross_sectional_areas(mask, skeleton, binary_edt, voxel_size_um=(1.0, 1.0, 1.0)):
    # Returns area_3d on skeleton voxels only.
    # Mathematical: ellipse area pi*(major/2)*(minor/2), with major from 2D EDT projection, minor from 3D EDT.
    # Biological: local vessel cross-sectional area proxy along centerlines.
    voxel_size_um = np.asarray(voxel_size_um, dtype=float)
    edt_2d = edt(np.max(mask, axis=0), sampling=(float(voxel_size_um[1]), float(voxel_size_um[2])))
    area_3d = np.zeros_like(binary_edt, dtype=float)
    z_idx, y_idx, x_idx = np.where(skeleton > 0)

    minor_axis = 2 * binary_edt[z_idx, y_idx, x_idx]
    major_axis = 2 * edt_2d[y_idx, x_idx]

    areas = np.pi * (major_axis / 2) * (minor_axis / 2)
    area_3d[z_idx, y_idx, x_idx] = areas
    return area_3d

def compute_vessel_metrics(graph, area_image, vessel_metrics_df, voxel_size_um=(1.0, 1.0, 1.0)):
    """
    Returns
    -------
    vessel_metrics_df : DataFrame indexed by (node1, node2) edge
        z, y, x: edge polyline coordinates (voxel path through volume).
        volume: sum of local cross-sectional areas along edge; vessel-volume surrogate.
        length: centerline arclength of edge.
        tortuosity: length / endpoint chord length; geometric winding (1=straight).
        is_sprout: True if edge touches an endpoint node (degree 1).
        median_cs_area / std_cs_area: cross-sectional area distribution along edge.
    """
    voxel_size_um = np.asarray(voxel_size_um, dtype=float)
    zs, ys, xs = [], [], []
    volumes, lengths = [], []
    tortuosities, is_sprouts = [], []
    median_cs_areas, std_cs_areas = [], []

    edges = list(graph.edges())
    valid_edges = []
    if not edges:
        cols = [
            'z', 'y', 'x', 'volume', 'length', 'tortuosity',
            'is_sprout', 'median_cs_area', 'std_cs_area'
        ]
        return pd.DataFrame(columns=cols)

    for u, v in edges:
        try:
            pts = graph[u][v]['pts']
            if len(pts) < 2:
                continue

            valid_edges.append((u, v))
            zs.append(pts[:, 0])
            ys.append(pts[:, 1])
            xs.append(pts[:, 2])

            segment_areas = area_image[pts[:, 0], pts[:, 1], pts[:, 2]]
            median_cs_areas.append(np.nanmedian(segment_areas))
            std_cs_areas.append(np.nanstd(segment_areas))

            pts_um = np.asarray(pts, dtype=float) * voxel_size_um[None, :]
            l = measure_edge_length(pts_um)
            lengths.append(l)
            sp = np.linalg.norm(pts_um[0] - pts_um[-1])
            tortuosities.append(np.clip(l / (sp + 1e-8), 0, 5))
            volumes.append(float(np.nanmean(segment_areas)) * float(l))

            is_sprouts.append(graph.nodes[u]['sprout'] or graph.nodes[v]['sprout'])
        except (KeyError, IndexError):
            continue

    vessel_metrics_df = pd.DataFrame(index=pd.MultiIndex.from_tuples(valid_edges, names=['node1', 'node2']))

    vessel_metrics_df['z'] = zs
    vessel_metrics_df['y'] = ys
    vessel_metrics_df['x'] = xs
    vessel_metrics_df['volume'] = volumes
    vessel_metrics_df['length'] = lengths
    vessel_metrics_df['tortuosity'] = tortuosities
    vessel_metrics_df['is_sprout'] = is_sprouts
    vessel_metrics_df['median_cs_area'] = median_cs_areas
    vessel_metrics_df['std_cs_area'] = std_cs_areas

    return vessel_metrics_df

def compute_junction_metrics(graph, junction_metrics_df, voxel_size_um=(1.0, 1.0, 1.0)):
    from heapq import heappush, heappop
    voxel_size_um = np.asarray(voxel_size_um, dtype=float)

    cols = ['z', 'y', 'x', 'number_of_vessel_per_node', 'node_type', 'dist_nearest_junction', 'dist_nearest_endpoint']
    nodes = list(graph.nodes())
    if not nodes:
        return pd.DataFrame(index=nodes, columns=cols)

    positions = np.array([graph.nodes[n]['pts'] for n in nodes])
    node_type = np.array(['sprout' if graph.nodes[n]['sprout'] else 'junction' for n in nodes])
    endpoint_nodes = set(np.array(nodes)[node_type == 'sprout'])
    junction_nodes = set(np.array(nodes)[node_type == 'junction'])

    junction_metrics_df = pd.DataFrame(index=nodes)
    junction_metrics_df['z'] = positions[:, 0]  # node z coordinate (depth location in tissue)
    junction_metrics_df['y'] = positions[:, 1]  # node y coordinate
    junction_metrics_df['x'] = positions[:, 2]  # node x coordinate
    junction_metrics_df['number_of_vessel_per_node'] = np.array([graph.degree[n] for n in nodes])  # topological node degree
    junction_metrics_df['node_type'] = node_type  # sprout=end node; junction=branching node
    junction_metrics_df['dist_nearest_junction'] = np.nan  # along-skeleton shortest-path arclength to nearest other junction
    junction_metrics_df['dist_nearest_endpoint'] = np.nan  # along-skeleton shortest-path arclength to nearest sprout endpoint

    if len(nodes) < 2:
        return junction_metrics_df

    adjacency = {n: [] for n in nodes}
    for u, v in graph.edges():
        edge_data = graph.get_edge_data(u, v, default={})
        pts = edge_data.get('pts', None)
        if pts is None or len(pts) < 2:
            # Enforce geodesic distances strictly along skeleton polylines.
            continue
        edge_len = float(measure_edge_length(np.asarray(pts, dtype=float) * voxel_size_um[None, :]))
        adjacency[u].append((v, edge_len))
        adjacency[v].append((u, edge_len))

    def dijkstra_from(source):
        dist = {n: np.inf for n in nodes}
        dist[source] = 0.0
        heap = [(0.0, source)]
        while heap:
            cur_dist, cur_node = heappop(heap)
            if cur_dist > dist[cur_node]:
                continue
            for nbr, w in adjacency[cur_node]:
                nd = cur_dist + w
                if nd < dist[nbr]:
                    dist[nbr] = nd
                    heappush(heap, (nd, nbr))
        return dist

    for node in nodes:
        dist_map = dijkstra_from(node)

        junction_targets = junction_nodes - {node}
        if junction_targets:
            junction_dists = np.array([dist_map[t] for t in junction_targets], dtype=float)
            junction_dists = junction_dists[np.isfinite(junction_dists)]
            if junction_dists.size > 0:
                junction_metrics_df.at[node, 'dist_nearest_junction'] = float(np.min(junction_dists))

        endpoint_targets = endpoint_nodes - {node}
        if endpoint_targets:
            endpoint_dists = np.array([dist_map[t] for t in endpoint_targets], dtype=float)
            endpoint_dists = endpoint_dists[np.isfinite(endpoint_dists)]
            if endpoint_dists.size > 0:
                junction_metrics_df.at[node, 'dist_nearest_endpoint'] = float(np.min(endpoint_dists))

    return junction_metrics_df

def fractal_dimension_and_lacunarity(binary, min_box_size=1, max_box_size=None, n_samples=12):
    """
    Box-counting FD and lacunarity on a binary mask.

    Input segmentation is assumed already cropped to foreground ROI.
    """
    binary = np.asarray(binary > 0)

    pts = np.argwhere(binary)
    if pts.size == 0:
        return np.nan, np.nan

    if max_box_size is None:
        max_box_size = int(np.floor(np.log2(np.min(binary.shape))))

    scales = np.floor(np.logspace(max_box_size, min_box_size, num=n_samples, base=2)).astype(np.int64)
    scales = np.unique(scales)
    scales = scales[scales > 0]

    log_inv_scale = []
    log_N = []
    lac_vals = []

    for s in scales:
        box_ids = pts // s
        unique_box_ids, counts = np.unique(box_ids, axis=0, return_counts=True)
        N = unique_box_ids.shape[0]
        if N < 2:
            continue

        log_inv_scale.append(np.log(1.0 / s))
        log_N.append(np.log(N))

        mass = counts.astype(float)

        if mass.size < 2:
            continue

        mu = float(np.mean(mass))
        lac_vals.append((float(np.var(mass)) / (mu * mu)) if mu > 0 else np.nan)

    if len(log_N) < 2:
        return np.nan, np.nan

    fd = float(np.polyfit(log_inv_scale, log_N, 1)[0])
    lac = float(np.nanmean(lac_vals)) if len(lac_vals) else np.nan
    # fd: scaling exponent of occupied box count (higher -> more space-filling complexity).
    # lac: mass heterogeneity across boxes (higher -> more uneven vessel/gap organization).
    return fd, lac

def graph2image(graph, shape):
    # Returns binary skeleton volume rebuilt from graph edges.
    # Biological: visualization-ready map of the final pruned vascular centerline network.
    pruned_skeleton = np.zeros(shape)
    for u, v in graph.edges():
        coords = graph.get_edge_data(u, v)['pts']

        clipped_coords = np.zeros_like(coords)
        clipped_coords[:, 0] = np.clip(coords[:, 0], 0, shape[0] - 1)
        clipped_coords[:, 1] = np.clip(coords[:, 1], 0, shape[1] - 1)
        clipped_coords[:, 2] = np.clip(coords[:, 2], 0, shape[2] - 1)

        pruned_skeleton[clipped_coords[:, 0], clipped_coords[:, 1], clipped_coords[:, 2]] = 1

    return pruned_skeleton

In [4]:
vasculature_segmentation= np.load(r"C:\Users\taylorhearn\git_repos\vascumap\working_outputs_double_crop\20250619_Vascumap_Dev25_11_Daisy10_20250619_Vascumap_Dev25_11_Daisy10_vessel_mask_iso_354_cropped.npy")
vasculature_segmentation = vasculature_segmentation[:, 50:vasculature_segmentation.shape[1]-250, 200:vasculature_segmentation.shape[1]-50]
from skimage.measure import label
vasculature_labels = label(vasculature_segmentation)
viewer = napari.Viewer()
viewer.add_labels(vasculature_segmentation)
viewer.add_labels(vasculature_labels)

c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


<Labels layer 'vasculature_labels' at 0x1b1eb79a9d0>

In [ ]:
##################### CLEAN SEGMENTATION #####################
voxel_size_um = (2.0, 2.0, 2.0)
chunk_size = (vasculature_segmentation.shape[0], 512, 512)
binary_smoothed = cupy_chunk_processing(volume=vasculature_segmentation.astype(np.float32), processing_func=ndi_gpu.gaussian_filter,sigma=3, chunk_size=chunk_size) > 0.5
binary_filled_holes = cupy_chunk_processing(volume=binary_smoothed, processing_func=ndi_gpu.binary_fill_holes, chunk_size=chunk_size).astype(np.float32)
binary_edt = cupy_chunk_processing(volume=binary_filled_holes, processing_func=ndi_gpu.distance_transform_edt, sampling=voxel_size_um, chunk_size=chunk_size)

##################### VESSEL MASK #####################
vessel_mask = binary_filled_holes.astype(bool)
masked_vessel_for_skeleton = vessel_mask

##################### SKELETONISE AND GRAPH #####################
dask_volume = da.from_array(masked_vessel_for_skeleton, chunks=chunk_size)
skeleton = da.overlap.map_overlap(dask_volume, skeletonize_3d, depth=(2, 2, 2), boundary='none', dtype=bool).compute(scheduler='threads')
graph = sknw.build_sknw(skeleton)
area_image = compute_cross_sectional_areas(vasculature_segmentation, skeleton, binary_edt, voxel_size_um=voxel_size_um) # local cross-sectional area in um^2

for n in list(graph.nodes()):
    graph.nodes[n]['pts'] = graph.nodes[n]['pts'][0]
    graph.nodes[n]['sprout'] = graph.degree(n) == 1

pruned_graph = prune_graph(graph=graph, area_3d=area_image, edt_cutoff=0.20, length_cutoff=25)
clean_graph = remove_mid_node(pruned_graph)
clean_graph = collect_border_vicinity_edges(clean_graph, vasculature_segmentation.shape)
isolated_nodes = [node for node in clean_graph.nodes() if clean_graph.degree[node] == 0]
if isolated_nodes:
    clean_graph.remove_nodes_from(isolated_nodes)

# Convert final cleaned graph back to a 3D skeleton volume
skeleton_from_graph = graph2image(clean_graph, vasculature_segmentation.shape).astype(np.uint8)
for node in clean_graph.nodes():
    clean_graph.nodes[node]['sprout'] = clean_graph.degree(node) == 1

##################### METRICS #####################
junction_metrics_df = pd.DataFrame()
vessel_metrics_df = pd.DataFrame()
global_metrics = {}

if clean_graph.number_of_edges() > 0:
    skeleton_mask = skeleton_from_graph > 0
    fd, skeleton_lacunarity = fractal_dimension_and_lacunarity(
        skeleton_mask,
    )
    _, vessel_lacunarity = fractal_dimension_and_lacunarity(
        vessel_mask,
    )
    total_vessel_length = np.sum([np.linalg.norm(np.diff(clean_graph[u][v]['pts'].astype(float) * np.asarray(voxel_size_um, dtype=float)[None, :], axis=0), axis=1).sum() for u, v in clean_graph.edges()])
    branchpoints_count = sum(1 for u in clean_graph.nodes() if not clean_graph.nodes[u]['sprout'])
    vessels_count = sum(1 for u, v in clean_graph.edges() if not clean_graph.nodes[u]['sprout'] and not clean_graph.nodes[v]['sprout'])
    sprouts_count = clean_graph.number_of_edges() - vessels_count
else:
    fd, skeleton_lacunarity, vessel_lacunarity = np.nan, np.nan, np.nan
    total_vessel_length = 0
    branchpoints_count = 0
    vessels_count = 0
    sprouts_count = 0

global_metrics['total_vessel_length'] = total_vessel_length  # sum of all centerline segment lengths
global_metrics['total_number_of_sprouts'] = sprouts_count  # edges connected to at least one endpoint
global_metrics['total_number_of_junctions'] = branchpoints_count  # count of non-endpoint nodes
global_metrics['fractal_dimension'] = fd  # skeleton box-counting complexity
global_metrics['skeleton_lacunarity'] = skeleton_lacunarity  # heterogeneity/patchiness on skeleton mask
global_metrics['vessel_lacunarity'] = vessel_lacunarity  # heterogeneity/patchiness on full vessel mask
global_metrics['lacunarity'] = skeleton_lacunarity  # backward-compatible alias for skeleton_lacunarity

if clean_graph.number_of_nodes() > 0 and clean_graph.number_of_edges() > 0:
    vessel_metrics_df = compute_vessel_metrics(clean_graph, area_image, vessel_metrics_df, voxel_size_um=voxel_size_um)
    junction_metrics_df = compute_junction_metrics(clean_graph, junction_metrics_df, voxel_size_um=voxel_size_um)

    if not vessel_metrics_df.empty:
        is_sprout_mask = vessel_metrics_df['is_sprout']
        global_metrics['median_sprout_and_branch_volume'] = float(np.nanmedian(vessel_metrics_df['volume'].to_numpy(dtype=float)))
        global_metrics['median_sprout_and_branch_tortuosity'] = float(np.nanmedian(vessel_metrics_df['tortuosity'].to_numpy(dtype=float)))
        global_metrics['median_sprout_and_branch_median_cs_area'] = float(np.nanmedian(vessel_metrics_df['median_cs_area'].to_numpy(dtype=float)))
        if np.any(~is_sprout_mask):
            global_metrics['median_branch_tortuosity'] = float(np.nanmedian(vessel_metrics_df.loc[~is_sprout_mask, 'tortuosity'].to_numpy(dtype=float)))
        else:
            global_metrics['median_branch_tortuosity'] = np.nan
        if np.any(is_sprout_mask):
            global_metrics['median_sprout_tortuosity'] = float(np.nanmedian(vessel_metrics_df.loc[is_sprout_mask, 'tortuosity'].to_numpy(dtype=float)))
        else:
            global_metrics['median_sprout_tortuosity'] = np.nan

    if not junction_metrics_df.empty:
        is_branch_point_mask = (junction_metrics_df['node_type'] == 'junction')
        is_sprout_tip_mask = (junction_metrics_df['node_type'] == 'sprout')
        global_metrics['median_junction_and_sprout_dist_nearest_junction'] = float(np.nanmedian(junction_metrics_df['dist_nearest_junction'].to_numpy(dtype=float)))
        global_metrics['median_junction_and_sprout_dist_nearest_endpoint'] = float(np.nanmedian(junction_metrics_df['dist_nearest_endpoint'].to_numpy(dtype=float)))
        if np.any(is_branch_point_mask):
            global_metrics['median_junction_dist_nearest_junction'] = float(np.nanmedian(junction_metrics_df.loc[is_branch_point_mask, 'dist_nearest_junction'].to_numpy(dtype=float)))
        else:
            global_metrics['median_junction_dist_nearest_junction'] = np.nan
        if np.any(is_sprout_tip_mask):
            global_metrics['median_sprout_dist_nearest_endpoint'] = float(np.nanmedian(junction_metrics_df.loc[is_sprout_tip_mask, 'dist_nearest_endpoint'].to_numpy(dtype=float)))
        else:
            global_metrics['median_sprout_dist_nearest_endpoint'] = np.nan

global_metrics = pd.DataFrame([global_metrics])

Processing chunks: 100%|██████████| 1/1 [00:01<00:00,  1.44s/it]
c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\dask\array\overlap.py:667: FutureWarning: The use of map_overlap(array, func, **kwargs) is deprecated since dask 2.17.0 and will be an error in a future release. To silence this warning, use the syntax map_overlap(func, array0,[ array1, ...,] **kwargs) instead.
  warnings.warn(


In [ ]:
viewer = napari.Viewer(ndisplay=3)
viewer.add_labels(vasculature_segmentation.astype(np.uint8), name="segmentation")
viewer.add_labels(binary_filled_holes.astype(np.uint8), name="filled")
viewer.add_labels(skeleton.astype(np.uint8), name="raw_skeleton")
viewer.add_labels(skeleton_from_graph, name="pruned_graph_skeleton")

# Optional: show graph nodes as points
node_pts = np.array([clean_graph.nodes[n]["pts"] for n in clean_graph.nodes()], dtype=float)
if len(node_pts) > 0:
    viewer.add_points(node_pts, name="graph_nodes")

In [31]:
global_metrics.head()

,total_vessel_length,total_number_of_sprouts,total_number_of_branches,total_number_of_junctions,fractal_dimension,lacunarity,mean_sprout_and_branch_volume,std_sprout_and_branch_volume,median_sprout_and_branch_volume,mean_branch_volume,...,median_sprout_num_junction_neighbors,mean_junction_and_sprout_num_endpoint_neighbors,std_junction_and_sprout_num_endpoint_neighbors,median_junction_and_sprout_num_endpoint_neighbors,mean_junction_num_endpoint_neighbors,std_junction_num_endpoint_neighbors,median_junction_num_endpoint_neighbors,mean_sprout_num_endpoint_neighbors,std_sprout_num_endpoint_neighbors,median_sprout_num_endpoint_neighbors
0,151814.829124,255,2522,1779,1.129302,0.324819,45864.392642,59298.31735,27160.825309,47417.931604,...,18.0,2.607564,2.089294,2.0,2.727937,2.10066,2.0,1.774319,1.800249,1.0


In [ ]:
clipped_vascualature_segmentation = vasculature_segmentation[:,200:2321, 200:1499 ]




In [77]:
from scipy import ndimage as ndi

def analyse_2d_holes_per_slice(mask, voxel_size_xy=1.0, min_hole_area_px=50, max_hole_area_fraction=0.15):
    """
    Quantify internal (border-excluded) holes independently in each z-slice.

    Mathematical definitions and biological interpretation:
    - area_px: number of pixels in a hole component (2D avascular pocket size in that slice).
    - area_um2: physical hole area (cross-sectional avascular area).
    - equivalent_radius_um: radius of a circle with same area (intuitive effective pore radius).
    - mean_radius_um / p90_radius_um / max_radius_um: EDT-based local half-width statistics
      inside each hole; higher values indicate wider local vessel-to-vessel gaps.

    Filtering:
    - min_hole_area_px removes very small holes/noise.
    - max_hole_area_fraction removes very large holes (for example border-like cavities)
      relative to each slice area, even after border exclusion.

    Additional outputs for visualization:
    - hole_labels_3d: integer labels of retained internal holes per slice (0=not a retained hole).
    - hole_distance_3d: EDT radius map inside retained holes; larger values = larger local pore width.
    """
    vessel = mask.astype(bool)
    z_depth = vessel.shape[0]

    hole_records = []
    centroid_records = []
    hole_labels_3d = np.zeros(mask.shape, dtype=np.int32)
    hole_distance_3d = np.zeros(mask.shape, dtype=np.float32)
    global_hole_id = 1

    for z in range(z_depth):
        vessel_slice = vessel[z]
        background = ~vessel_slice
        slice_area_px = int(vessel_slice.size)
        max_hole_area_px = float(max_hole_area_fraction) * float(slice_area_px)

        # Keep only internal holes: remove background component(s) connected to image border.
        lbl_bg, _ = ndi.label(background, structure=np.ones((3, 3), dtype=np.uint8))
        if lbl_bg.max() == 0:
            continue

        border_labels = np.unique(np.concatenate([
            lbl_bg[0, :], lbl_bg[-1, :], lbl_bg[:, 0], lbl_bg[:, -1]
        ]))

        internal_holes = background.copy()
        for b_lab in border_labels:
            if b_lab != 0:
                internal_holes[lbl_bg == b_lab] = False

        # Label per-slice internal holes for per-component measurements.
        lbl_holes, n_holes = ndi.label(internal_holes, structure=np.ones((3, 3), dtype=np.uint8))
        if n_holes == 0:
            continue

        edt_holes = ndi.distance_transform_edt(internal_holes, sampling=(voxel_size_xy, voxel_size_xy))

        for hole_id in range(1, n_holes + 1):
            comp = lbl_holes == hole_id
            area_px = int(np.count_nonzero(comp))
            if area_px < int(min_hole_area_px):
                continue
            if area_px > max_hole_area_px:
                continue

            area_um2 = float(area_px) * float(voxel_size_xy) * float(voxel_size_xy)
            equivalent_radius_um = float(np.sqrt(area_um2 / np.pi))

            radii = edt_holes[comp]
            mean_radius_um = float(np.mean(radii)) if radii.size else np.nan
            p90_radius_um = float(np.percentile(radii, 90)) if radii.size else np.nan
            max_radius_um = float(np.max(radii)) if radii.size else np.nan

            yy, xx = np.where(comp)
            centroid_y = float(np.mean(yy))
            centroid_x = float(np.mean(xx))

            # Persist retained holes into 3D overlay volumes.
            hole_labels_3d[z][comp] = global_hole_id
            hole_distance_3d[z][comp] = edt_holes[comp].astype(np.float32)
            this_global_id = global_hole_id
            global_hole_id += 1

            hole_records.append({
                "z": int(z),
                "hole_id": int(this_global_id),
                "area_px": area_px,
                "area_um2": area_um2,
                "equivalent_radius_um": equivalent_radius_um,
                "mean_radius_um": mean_radius_um,
                "p90_radius_um": p90_radius_um,
                "max_radius_um": max_radius_um,
            })

            centroid_records.append({
                "z": int(z),
                "y": centroid_y,
                "x": centroid_x,
                "equivalent_radius_um": equivalent_radius_um,
                "mean_radius_um": mean_radius_um,
            })

    holes_df = pd.DataFrame(hole_records)
    centroids_df = pd.DataFrame(centroid_records)

    if holes_df.empty:
        slice_summary_df = pd.DataFrame(columns=[
            "z", "n_holes", "mean_area_um2", "median_area_um2", "mean_equivalent_radius_um", "p90_equivalent_radius_um"
        ])
        global_summary_df = pd.DataFrame([{
            "n_slices_with_holes": 0,
            "total_holes": 0,
            "mean_holes_per_active_slice": np.nan,
            "mean_equivalent_radius_um": np.nan,
            "p90_equivalent_radius_um": np.nan,
        }])
        return holes_df, slice_summary_df, global_summary_df, centroids_df, hole_labels_3d, hole_distance_3d

    slice_summary_df = (
        holes_df.groupby("z", as_index=False)
        .agg(
            n_holes=("hole_id", "count"),
            mean_area_um2=("area_um2", "mean"),
            median_area_um2=("area_um2", "median"),
            mean_equivalent_radius_um=("equivalent_radius_um", "mean"),
            p90_equivalent_radius_um=("equivalent_radius_um", lambda v: float(np.percentile(v, 90))),
        )
    )

    global_summary_df = pd.DataFrame([{
        "n_slices_with_holes": int(slice_summary_df.shape[0]),
        "total_holes": int(holes_df.shape[0]),
        "mean_holes_per_active_slice": float(slice_summary_df["n_holes"].mean()),
        "mean_equivalent_radius_um": float(holes_df["equivalent_radius_um"].mean()),
        "p90_equivalent_radius_um": float(np.percentile(holes_df["equivalent_radius_um"], 90)),
    }])

    return holes_df, slice_summary_df, global_summary_df, centroids_df, hole_labels_3d, hole_distance_3d


slice_holes_df, slice_hole_stats_df, hole_global_summary_df, slice_hole_centroids_df, slice_hole_labels_3d, slice_hole_distance_3d = analyse_2d_holes_per_slice(
    clipped_vascualature_segmentation,
    voxel_size_xy=1.0,
    min_hole_area_px=50,
    max_hole_area_fraction=0.25,
 )

print("2D internal holes detected:", len(slice_holes_df))
print(hole_global_summary_df.to_string(index=False))
display(slice_hole_stats_df.head(15))
display(slice_holes_df.sort_values(["z", "area_um2"], ascending=[True, False]).head(20))

2D internal holes detected: 14876
 n_slices_with_holes  total_holes  mean_holes_per_active_slice  mean_equivalent_radius_um  p90_equivalent_radius_um
                  49        14876                   303.591837                  17.507592                 35.727055


,z,n_holes,mean_area_um2,median_area_um2,mean_equivalent_radius_um,p90_equivalent_radius_um
0,0,237,1227.417722,438.0,15.347728,31.075500
1,1,241,1240.643154,407.0,15.150192,29.332433
2,2,245,1332.812245,403.0,15.683431,31.008174
3,3,251,1438.828685,427.0,16.412751,32.361080
4,4,250,1510.236000,514.5,17.281088,33.142443
5,5,253,1538.565217,526.0,17.439071,33.596518
6,6,257,1450.245136,592.0,17.334519,33.627284
7,7,275,1562.247273,567.0,17.353359,34.139262
8,8,273,1567.783883,598.0,17.633249,34.274622
9,9,269,1571.591078,612.0,17.936982,34.951875


,z,hole_id,area_px,area_um2,equivalent_radius_um,mean_radius_um,p90_radius_um,max_radius_um
53,0,54,26555,26555.0,91.938670,9.973801,22.000000,41.231056
178,0,179,12425,12425.0,62.888793,17.161562,34.664102,49.477268
121,0,122,11107,11107.0,59.459801,9.393989,19.697716,31.240999
0,0,1,9379,9379.0,54.639074,8.614292,18.027756,29.154759
80,0,81,9004,9004.0,53.535616,8.486372,16.124515,24.351591
77,0,78,8848,8848.0,53.069821,5.617077,11.661904,20.024984
122,0,123,8204,8204.0,51.101999,7.387146,16.278821,27.000000
220,0,221,8104,8104.0,50.789599,6.381258,12.000000,18.867962
14,0,15,6854,6854.0,46.708628,8.883896,17.492856,23.259407
19,0,20,6399,6399.0,45.131640,9.589784,18.384776,25.495098


In [78]:
viewer = napari.Viewer(ndisplay=3)
viewer.add_labels(clipped_vascualature_segmentation.astype(np.uint8), name="segmentation", opacity=0.45)

if np.any(slice_hole_labels_3d > 0):
    # Discrete retained holes (per-slice components).
    viewer.add_labels(slice_hole_labels_3d.astype(np.int32), name="2D_internal_holes_labels", opacity=0.55)

# Continuous local pore-width map (EDT radius inside retained holes).
positive_dist = slice_hole_distance_3d[slice_hole_distance_3d > 0]
if positive_dist.size > 0:
    vmax = float(np.percentile(positive_dist, 99))
    viewer.add_image(
        slice_hole_distance_3d.astype(np.float32),
        name="2D_internal_holes_distance_um",
        colormap="magma",
        blending="additive",
        opacity=0.6,
        contrast_limits=(0.0, max(vmax, 1e-6)),
    )

if len(slice_hole_centroids_df) > 0:
    hole_points = slice_hole_centroids_df[["z", "y", "x"]].to_numpy(dtype=float)
    hole_sizes = np.clip(2.0 * slice_hole_centroids_df["equivalent_radius_um"].to_numpy(dtype=float), 1.0, 30.0)
    viewer.add_points(
        hole_points,
        name="2D_internal_hole_centroids",
        size=hole_sizes,
        face_color="cyan",
        border_color="white",
        opacity=0.9,
    )

print(f"2D internal holes overlaid: {len(slice_hole_centroids_df)}")

2D internal holes overlaid: 14876
